# 3 · Aprendizaje continuo y deriva

El nodo **ia aprender** de Node-RED entrena un modelo **mientras llegan los datos** y avisa de **deriva** cuando
su error empeora de golpe. El flujo de la fase 3 guarda sus resultados en el conjunto `aerolab-continuo`.

Aquí comparamos ese modelo continuo con el modelo **fijo** que entrenaste en el cuaderno 2.

In [ ]:
from labia import datos, cargar, predecir
import matplotlib.pyplot as plt

dc = datos("aerolab-continuo")
fijo, ficha = cargar("aerolab-curva")
dc["fijo"] = [r["prediccion"] for r in predecir(fijo, ficha, dc[["viento"]].to_dict("records"))]
dc[["ts", "escenario", "viento", "potencia", "continuo_prediccion", "fijo", "continuo_mae", "continuo_deriva"]].tail()

In [ ]:
fig, (a1, a2) = plt.subplots(2, 1, figsize=(11, 7), sharex=True)
a1.plot(dc.ts, (dc.potencia - dc.fijo).abs().rolling(60).mean(), label="error del modelo fijo")
a1.plot(dc.ts, (dc.potencia - dc.continuo_prediccion).abs().rolling(60).mean(), label="error del modelo continuo")
a1.set_ylabel("error medio (kW, 60 s)"); a1.legend(); a1.grid(alpha=.3)
a2.plot(dc.ts, dc.potencia, lw=.7, label="potencia")
for t in dc.ts[dc.continuo_deriva == 1]:
    for ax in (a1, a2):
        ax.axvline(t, color="red", alpha=.6)
a2.set_ylabel("kW"); a2.grid(alpha=.3); a2.set_title("En rojo, las derivas detectadas");

In [ ]:
dc.groupby("escenario")[["continuo_mae"]].mean().round(0)

## ✏️ Para pensar

1. Cuando se limita la potencia a 3000 kW, ¿qué modelo se equivoca más? ¿Cuánto tarda el continuo en adaptarse?
2. Con la góndola desalineada, el modelo continuo también se adapta… **¿es bueno?** ¿Qué perderías si dejaras
   que un modelo aprenda siempre sin supervisión?
3. Propón una regla: ¿cuándo debe aprender el modelo y cuándo no? (pista: el flujo ya no le deja aprender con
   avería; ¿qué más filtrarías?)